# Hypothesis Testing
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** the null and alternative hypotheses and explain what each one claims
- **Describe** the logic of hypothesis testing as a decision made under uncertainty
- **Interpret** the rejection region and significance level as a threshold for calling a result "statistically significant"

> **Tip:** Start by moving the **effect size slider** toward zero, watch the test statistic move into the "fail to reject" zone and see why small effects are hard to detect.

---
## How we got here

In *16: Confidence Intervals* we built a range that captures the true parameter with known probability. Hypothesis testing flips this: instead of estimating where the parameter is, we start with a specific claim (the null hypothesis) and ask whether our data is consistent with it or whether it is too extreme to be explained by chance alone.

---
## Why this matters for data science

Hypothesis testing is the engine of A/B testing, every product decision that compares two versions of a feature, algorithm, or UI element uses this framework. Understanding the null hypothesis, the test statistic, and the rejection region is the foundation for interpreting any experiment result, and misunderstanding it is one of the most common sources of bad decisions in data-driven organizations.

---
## Try it yourself

In [2]:
from tkh_utils import make_slider, make_int_slider, make_dropdown, make_output, display_widget, register_observer

_x_range = np.linspace(-4, 4, 500)

out = make_output()
delta_slider   = make_slider(0.0, 3.0, 0.1, 1.0, "Effect size (δ):")
n_slider       = make_int_slider(5, 200, 5, 30, "Sample size (n):")
alpha_dropdown = make_dropdown(["0.01", "0.05", "0.10"], "0.05", "Alpha (α):")

def render(delta, n, alpha_str):
    alpha  = float(alpha_str)
    se     = 1.0 / np.sqrt(n)
    z_obs  = delta / se
    z_crit = stats.norm.ppf(1 - alpha)
    decision = "Reject H₀" if z_obs > z_crit else "Fail to reject H₀"

    y_curve = stats.norm.pdf(_x_range)
    x_rej   = _x_range[_x_range >= z_crit]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=_x_range, y=y_curve,
        mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name="Null distribution",
    ))
    if len(x_rej):
        fig.add_trace(go.Scatter(
            x=np.concatenate([[x_rej[0]], x_rej, [x_rej[-1]]]),
            y=np.concatenate([[0], stats.norm.pdf(x_rej), [0]]),
            fill="toself",
            fillcolor="rgba(247,103,7,0.30)",
            line=dict(width=0),
            name=f"Rejection region (α = {alpha})",
        ))
    z_plot = min(z_obs, 4.0)
    fig.add_vline(
        x=z_plot,
        line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
        annotation_text=f"z_obs = {z_obs:.2f}",
    )
    fig.add_vline(
        x=z_crit,
        line_color=PALETTE["muted"], line_width=1.5, line_dash="dot",
        annotation_text=f"z_crit = {z_crit:.2f}",
    )
    layout = base_layout(
        title=f"Hypothesis Test — z = {z_obs:.2f}, z_crit = {z_crit:.2f} | {decision}",
        xaxis_title="Test Statistic (z)",
        yaxis_title="Probability Density under H₀",
    )
    layout.update(xaxis=dict(range=[-4, 4]))
    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

register_observer(
    [delta_slider, n_slider, alpha_dropdown],
    lambda change: render(delta_slider.value, n_slider.value, alpha_dropdown.value),
)
display_widget([delta_slider, n_slider, alpha_dropdown], out)
render(delta_slider.value, n_slider.value, alpha_dropdown.value)


---
## What's happening?

Hypothesis testing is a formal procedure for deciding whether data provides enough evidence to reject a default assumption (the null hypothesis).

| Element | Symbol | Meaning |
|---------|--------|---------|
| Null hypothesis | H₀ | The "no effect" or "status quo" claim |
| Alternative hypothesis | H₁ or Hₐ | What you are trying to demonstrate |
| Significance level | α | Threshold probability for rejecting H₀ (usually 0.05) |
| Test statistic | z or t | Standardized measure of how far the data is from H₀ |
| Rejection region | — | Values of the test statistic extreme enough to reject H₀ |

The general form of a one-sample z-test:

$$z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$$

### The logic of the test

We assume H₀ is true, then ask: "How likely is it to see data this extreme or more extreme, by chance alone?" If that probability (the p-value, covered in the next notebook) is below α, we conclude the data is too unlikely under H₀, and we reject it.

Return to the widget and set effect size to zero, the test statistic lands at 0, squarely in the "fail to reject" zone. Only real effects push it into the tails.

---
## Real-world example: A/B testing a website headline

A product team wants to know whether changing a homepage headline increases the click-through rate (CTR). They run an experiment: 1,000 users see the old headline (control), 1,000 see the new one (treatment). The question: is the observed difference in CTR real, or could it have occurred by chance?

The chart shows the null distribution (what differences we would expect if there were no effect) alongside the observed test statistic. Notice:

- **Notice:** The null distribution is centered at zero, under H₀, the two headlines perform equally, so any difference is just sampling noise
- **Notice:** The observed difference falls in the right tail, past the critical value, this is the rejection region
- **Notice:** The shaded area in the tail represents the probability of seeing a difference this large (or larger) by chance, the smaller this area, the stronger the evidence against H₀

> **Discussion question:** The team observes a 0.8 percentage point improvement in CTR. Is this "statistically significant" the same as "practically important"? What else would you want to know before recommending the change?

In [3]:
np.random.seed(303)

# ── A/B test: click-through rate experiment ─────────────────────────────────────
n         = 1000
p_control = 0.05          # 5% baseline CTR
p_treat   = 0.058         # 5.8% treatment CTR — 0.8pp lift
alpha     = 0.05

clicks_c  = np.random.binomial(n, p_control)
clicks_t  = np.random.binomial(n, p_treat)
diff_obs  = clicks_t / n - clicks_c / n

# Pooled proportion under H0
p_pool  = (clicks_c + clicks_t) / (2 * n)
se_null = np.sqrt(2 * p_pool * (1 - p_pool) / n)
z_obs   = diff_obs / se_null
z_crit  = stats.norm.ppf(1 - alpha)

x_z = np.linspace(-4, 4, 500)
y_z = stats.norm.pdf(x_z)

x_rej = x_z[x_z >= z_crit]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_z, y=y_z, mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5),
    name="Null distribution",
))
fig.add_trace(go.Scatter(
    x=np.concatenate([[z_crit], x_rej, [x_rej[-1]]]),
    y=np.concatenate([[0], stats.norm.pdf(x_rej), [0]]),
    fill="toself", fillcolor="rgba(247,103,7,0.25)",
    line=dict(color=PALETTE["secondary"], width=0),
    name=f"Rejection region (α={alpha})",
))
fig.add_vline(x=z_obs, line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
              annotation_text=f"z_obs = {z_obs:.2f}")
fig.add_vline(x=z_crit, line_color=PALETTE["muted"], line_width=1.5, line_dash="dot",
              annotation_text=f"z_crit = {z_crit:.2f}")
layout = base_layout(
    title=f"Hypothesis Test: A/B CTR Experiment  (z = {z_obs:.2f}, α = {alpha})",
    xaxis_title="Test Statistic (z)",
    yaxis_title="Probability Density under H₀",
)
layout.update(xaxis=dict(range=[-4, 4]))
fig.update_layout(layout)
fig.show()

### The four-step hypothesis testing procedure

| Step | Action | Example |
|------|--------|---------|
| 1. State hypotheses | Write H₀ and H₁ clearly | H₀: CTR_new = CTR_old; H₁: CTR_new > CTR_old |
| 2. Choose α | Set the acceptable false-alarm rate before seeing data | α = 0.05 |
| 3. Compute test statistic | Standardize the observed result under H₀ | z = (x̄ − μ₀) / SE |
| 4. Make decision | Compare test statistic to critical value (or p-value to α) | Reject H₀ if z > z_crit |

---
## Key takeaway

> **Hypothesis testing asks whether data is too extreme to be explained by chance; the significance level α sets the tolerance for being wrong — and a smaller α demands stronger evidence.**

---
*Next up: P-Values — the probability that connects the test statistic to the binary reject/fail-to-reject decision*